In [5]:
import numpy as np
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold, cross_val_score, cross_val_predict, train_test_split
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, accuracy_score, classification_report
import torch
import torch.nn as nn
from torch.autograd import Variable
from torch.utils.data import Dataset, DataLoader

load_previously_saved_model = 0

In [6]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE

'cpu'

In [7]:
data = pd.read_csv('Titanic-Dataset.csv')

In [8]:
data.drop(columns = ['Name', 'PassengerId',  'Cabin','SibSp', 'Parch', 'Ticket', 'Pclass'], inplace = True)

categorical_features = ['Sex', 'Embarked', 'Survived']
numeric_features = [x for x in list(data.columns) if x not in categorical_features]
#numeric_features = ['latitude', 'longitude', 'price', 'minimum_nights']


data[numeric_features] = data[numeric_features].fillna(data[numeric_features].median())

data[categorical_features] = data[categorical_features].fillna(data[categorical_features].mode().iloc[0])
data = pd.get_dummies(data, columns = categorical_features, dtype=int)

sc = StandardScaler()
data[numeric_features] = sc.fit_transform(data[numeric_features])

y = data[['Survived_0', 'Survived_1']]
X = data.drop(columns = ['Survived_0', 'Survived_1'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=42)

dataset_train = pd.concat([X_train, y_train], axis = 1)

In [9]:
# sc_x = StandardScaler()
# X_train[numeric_features] = sc_x.fit_transform(X_train[numeric_features])
# X_test[numeric_features] = sc_x.transform(X_test[numeric_features])

# sc_y = StandardScaler()
# y_train = sc_y.fit_transform(y_train)
# y_test = sc_y.transform(y_test)

In [10]:
def accuracy(outputs, labels):
    
    _, preds = torch.max(outputs, dim=1)
    y_test['Survived'] = labels.apply(lambda x: 0 if x['Survived_0'] == 1 else 1, axis=1)
    out = torch.tensor(y_test['Survived'].values)
       
    return preds, out

In [11]:
class LanguageModelDataLoader(DataLoader):
    def __init__(self, X_train, y_train, batch_size, shuffle=True):
        self.batch_size = batch_size
        self.cuda = False
        self.X_train = X_train
        self.y_train = y_train

    def __iter__(self):
        # concatenate your articles and build into batches
        count = len(self.X_train)
        num_batches = (count - 1)//self.batch_size
        number = num_batches * self.batch_size
        
        x = torch.from_numpy(self.X_train.values)
        y = torch.from_numpy(self.y_train.values)
        if self.cuda:
            x = x.cuda()
            y = y.cuda()
        
        start = 0
        while start < num_batches:
            # if (np.random.uniform() < 0.95):
            #     L = round(np.random.normal(70,5))
            # else:
            #     L = round(np.random.normal(35,5))
            L = self.batch_size
            x_out = x[start:start + L]
            y_out = y[start:start + L]
            #x_out = x_out.reshape(x_out.shape[0], 1, x_out.shape[1])
            start = start + L
            #print(x_out.shape, y_out.shape)
            yield x_out, y_out 

In [12]:
class LanguageModel(nn.Module):
    def __init__(self, dropout = 0, batch_norm = 0):
        super(LanguageModel, self).__init__()
        # self.vocab_size = charcount
        # self.embedding = nn.Embedding(self.vocab_size,400)
        self.linear = nn.ModuleList()
        self.linear.append(nn.Linear(7,64))
        if batch_norm:
            self.linear.append(nn.BatchNorm1d(64))
        self.linear.append(nn.ReLU())
        self.linear.append(nn.Linear(64,32))
        if batch_norm:
            self.linear.append(nn.BatchNorm1d(32))
        self.linear.append(nn.ReLU())
        if dropout:
            self.linear.append(nn.Dropout(p = 0.3))
        self.linear.append(nn.Linear(32,2))
        
    def forward(self, out):
        for layers in self.linear:
            out = layers(out)
        return out

In [13]:
class LanguageModelTrainer:
    def __init__(self, model, loader, X_test, y_test, max_epochs=1, run_id='exp'):
        # feel free to add any other parameters here
        self.model = model
        self.loader = loader
        self.predictions = []
        self.epochs = 0
        self.max_epochs = max_epochs
        self.run_id = run_id
        self.sigmoid = nn.Sigmoid()
        self.train_losses = []
        self.val_losses = []
        
        self.X_test = X_test
        self.y_test = torch.from_numpy(y_test[['Survived_0', 'Survived_1']].values)
        
        # TODO: Define your optimizer and criterion here
        self.optimizer = torch.optim.Adam(model.parameters(),lr=0.001, weight_decay=1e-6)
        self.criterion = nn.BCELoss().to(DEVICE)

    def train(self):
        self.model.train() # set to training mode
        epoch_loss = 0
        num_batches = 0
        for batch_num, (inputs, targets) in enumerate(self.loader):
            epoch_loss += self.train_batch(inputs, targets)
        epoch_loss = epoch_loss / (batch_num + 1)
        predictions = self.sigmoid(torch.from_numpy(TestLanguageModel.prediction(np.array(self.X_test), self.model)))
        #preds, score = accuracy(predictions, self.y_test)
        val_loss = self.criterion(predictions.to(torch.float32), self.y_test.to(torch.float32))
        self.epochs += 1
        if self.epochs % 25 == 0:
            print('[TRAIN]  Epoch [%d/%d]   Loss: %.4f [VALID LOSS] %.4f'
                      % (self.epochs, self.max_epochs, epoch_loss.item(), val_loss))
        self.train_losses.append(epoch_loss.item())
        self.val_losses.append(val_loss)
        return val_loss

    def train_batch(self, inputs, targets):
        inputs = inputs.to(DEVICE).long()
        targets = targets.to(torch.float32)
        outputs = self.model(inputs.to(torch.float32))
        outputs = self.sigmoid(outputs)
        # print("pre output is", outputs)
        # print("Post output is", outputs.view(-1, 2))
        # print("Target is", targets)
        loss = self.criterion(outputs.view(-1,2), targets)
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        return loss

In [14]:
class TestLanguageModel:
    def prediction(inp, model):
        model.eval()
        inp = torch.from_numpy(inp)
        inp = inp.to(torch.float32)
        h = model(inp)
        return h.cpu().data.numpy()

In [15]:
#hyperparameters here

NUM_EPOCHS = 50
BATCH_SIZE = 64

In [16]:
import mlflow
import mlflow.pytorch

mlflow.set_experiment("Day27_Regularisation_Comparison")
mlflow.set_tracking_uri("http://127.0.0.1:5000/")
last_layer = nn.Sigmoid()

configs = [
    {"dropout": 0, "batch_norm": 0, "run_name": "Baseline"},
    {"dropout": 0, "batch_norm": 1, "run_name": "BatchNorm_Only"},
    {"dropout": 1, "batch_norm": 0, "run_name": "Dropout_Only"},
    {"dropout": 1, "batch_norm": 1, "run_name": "BatchNorm_And_Dropout"},
]

for config in configs:
    with mlflow.start_run(run_name=config["run_name"]):
        
        # Log hyperparameters
        mlflow.log_param("dropout",    config["dropout"])
        mlflow.log_param("batch_norm", config["batch_norm"])
        mlflow.log_param("num_epochs", NUM_EPOCHS)
        mlflow.log_param("batch_size", BATCH_SIZE)
        mlflow.log_param("lr",         0.001)
        
        # Train the model
        model   = LanguageModel(
                    dropout=config["dropout"], 
                    batch_norm=config["batch_norm"]
                  )
        loader  = LanguageModelDataLoader(
                    X_train=X_train, y_train=y_train, 
                    batch_size=BATCH_SIZE
                  )
        trainer = LanguageModelTrainer(
                    model=model, loader=loader,
                    max_epochs=NUM_EPOCHS,
                    X_test=X_test, y_test=y_test
                  )
        
        for epoch in range(NUM_EPOCHS):
            trainer.train()
            
            # Log loss every epoch so MLflow plots the curve
            mlflow.log_metric("train_loss", trainer.train_losses[-1], step=epoch)
            mlflow.log_metric("val_loss",   float(trainer.val_losses[-1]),  step=epoch)
        
        # Final metrics
        preds = last_layer(torch.from_numpy(TestLanguageModel.prediction(X_test.values, model)))
        preds, out = accuracy(preds, y_test)
        acc = accuracy_score(y_test['Survived'].values, preds.numpy())
        report = classification_report(y_test['Survived'].values, preds.numpy(), output_dict = True)

        mlflow.log_metric("Accuracy", acc)
        mlflow.log_metric("F1-Score", report['weighted avg']['f1-score'])
        
        # Log the model
        mlflow.pytorch.log_model(model, config["run_name"], serialization_format='pt2')
        
        print(f"{config['run_name']}")
        print(report)

[TRAIN]  Epoch [25/50]   Loss: 0.6433 [VALID LOSS] 0.6544
[TRAIN]  Epoch [50/50]   Loss: 0.5483 [VALID LOSS] 0.5717


2026/05/07 11:10:50 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/07 11:10:50 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.


Baseline
{'0': {'precision': 0.7272727272727273, 'recall': 0.9230769230769231, 'f1-score': 0.8135593220338984, 'support': 78.0}, '1': {'precision': 0.8285714285714286, 'recall': 0.5178571428571429, 'f1-score': 0.6373626373626373, 'support': 56.0}, 'accuracy': 0.753731343283582, 'macro avg': {'precision': 0.7779220779220779, 'recall': 0.7204670329670331, 'f1-score': 0.7254609796982678, 'support': 134.0}, 'weighted avg': {'precision': 0.769606512890095, 'recall': 0.753731343283582, 'f1-score': 0.7399248866488938, 'support': 134.0}}
🏃 View run Baseline at: http://127.0.0.1:5000/#/experiments/1/runs/ee4ecad46a5c4875b770f5289cba4c91
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
[TRAIN]  Epoch [25/50]   Loss: 0.5386 [VALID LOSS] 0.5980
[TRAIN]  Epoch [50/50]   Loss: 0.4700 [VALID LOSS] 0.5679


2026/05/07 11:11:41 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/07 11:11:42 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.


BatchNorm_Only
{'0': {'precision': 0.7, 'recall': 0.8974358974358975, 'f1-score': 0.7865168539325843, 'support': 78.0}, '1': {'precision': 0.7647058823529411, 'recall': 0.4642857142857143, 'f1-score': 0.5777777777777777, 'support': 56.0}, 'accuracy': 0.7164179104477612, 'macro avg': {'precision': 0.7323529411764705, 'recall': 0.6808608058608059, 'f1-score': 0.682147315855181, 'support': 134.0}, 'weighted avg': {'precision': 0.7270412642669006, 'recall': 0.7164179104477612, 'f1-score': 0.6992826131514711, 'support': 134.0}}
🏃 View run BatchNorm_Only at: http://127.0.0.1:5000/#/experiments/1/runs/59cef078be71492da460a213f32b772f
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
[TRAIN]  Epoch [25/50]   Loss: 0.6625 [VALID LOSS] 0.6571
[TRAIN]  Epoch [50/50]   Loss: 0.5770 [VALID LOSS] 0.5817


2026/05/07 11:12:10 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/07 11:12:11 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.


Dropout_Only
{'0': {'precision': 0.782608695652174, 'recall': 0.9230769230769231, 'f1-score': 0.8470588235294118, 'support': 78.0}, '1': {'precision': 0.8571428571428571, 'recall': 0.6428571428571429, 'f1-score': 0.7346938775510204, 'support': 56.0}, 'accuracy': 0.8059701492537313, 'macro avg': {'precision': 0.8198757763975155, 'recall': 0.7829670329670331, 'f1-score': 0.7908763505402161, 'support': 134.0}, 'weighted avg': {'precision': 0.8137573004542504, 'recall': 0.8059701492537313, 'f1-score': 0.8001003386429197, 'support': 134.0}}
🏃 View run Dropout_Only at: http://127.0.0.1:5000/#/experiments/1/runs/71cd16ea626f4e8998839e66503aab71
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
[TRAIN]  Epoch [25/50]   Loss: 0.5334 [VALID LOSS] 0.5976
[TRAIN]  Epoch [50/50]   Loss: 0.4686 [VALID LOSS] 0.5855


2026/05/07 11:12:43 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/07 11:12:44 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.


BatchNorm_And_Dropout
{'0': {'precision': 0.696969696969697, 'recall': 0.8846153846153846, 'f1-score': 0.7796610169491526, 'support': 78.0}, '1': {'precision': 0.7428571428571429, 'recall': 0.4642857142857143, 'f1-score': 0.5714285714285714, 'support': 56.0}, 'accuracy': 0.7089552238805971, 'macro avg': {'precision': 0.71991341991342, 'recall': 0.6744505494505495, 'f1-score': 0.6755447941888619, 'support': 134.0}, 'weighted avg': {'precision': 0.7161465400271371, 'recall': 0.7089552238805971, 'f1-score': 0.6926385024032381, 'support': 134.0}}
🏃 View run BatchNorm_And_Dropout at: http://127.0.0.1:5000/#/experiments/1/runs/9c903d9b4e014923b44e5a63ad9ac089
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
